<a href="https://colab.research.google.com/github/niranjini2607-crypto/chip-failure-ai/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip archive.zip

Archive:  archive.zip
  inflating: uci-secom.csv           


In [ ]:
import pandas as pd

df = pd.read_csv("uci-secom.csv")

print("Shape:", df.shape)
print(df.columns[:3].tolist(), "...", df.columns[-3:].tolist())
print(df.tail(3).iloc[:, -3:])

Shape: (1567, 592)
['Time', '0', '1'] ... ['588', '589', 'Pass/Fail']
         588       589  Pass/Fail
1564  0.0025   43.5231         -1
1565  0.0075   93.4941         -1
1566  0.0045  137.7844         -1


In [ ]:
# 1. Pass vs Fail count
print("Label counts:")
print(df["Pass/Fail"].value_counts())
print()

fail_pct = (df["Pass/Fail"] == 1).mean() * 100
print(f"Fail percentage: {fail_pct:.2f}%")
print()

# 2. Missing values
sensors = df.drop(columns=["Time", "Pass/Fail"])
print("Overall missing %:", round(sensors.isnull().mean().mean() * 100, 2))
print("Columns with >50% missing:", (sensors.isnull().mean() > 0.5).sum())
print("Columns with any missing:", (sensors.isnull().sum() > 0).sum())
print()

# 3. Constant columns (no useful info)
constant_cols = sensors.columns[sensors.nunique() <= 1]
print("Constant columns:", len(constant_cols))

Label counts:
Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

Fail percentage: 6.64%

Overall missing %: 4.54
Columns with >50% missing: 28
Columns with any missing: 538

Constant columns: 116


Finding	Value	Meaning
Total samples	1567
Pass / Fail	1463 / 104	Fail 6.64% mattum, class imbalance
Overall missing	4.54%	Imputation venum
Columns >50% missing	28	Idha drop pannalam
Columns with any missing	538 / 590	Almost ellam la konjam missing
Constant columns	116	Information illa, remove pannanum

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Label 0/1 ku maathu (1 = fail)
y = (df["Pass/Fail"] == 1).astype(int)

# 2. Features: Time and label remove
X = df.drop(columns=["Time", "Pass/Fail"])

# 3. Drop >50% missing and constant columns
missing_cols = X.columns[X.isnull().mean() > 0.5]
constant_cols = X.columns[X.nunique() <= 1]
drop_cols = set(missing_cols) | set(constant_cols)
X = X.drop(columns=list(drop_cols))
print("Dropped columns:", len(drop_cols))
print("Remaining features:", X.shape[1])

# 4. Train/test split (stratify = fail ratio rendulayum same irukkum)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5. Impute (median) and scale - train la mattum fit pannanum
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc = scaler.transform(X_test_imp)

print("Train shape:", X_train_sc.shape, "| Fails:", y_train.sum())
print("Test shape:", X_test_sc.shape, "| Fails:", y_test.sum())

Dropped columns: 144
Remaining features: 446
Train shape: (1253, 446) | Fails: 83
Test shape: (314, 446) | Fails: 21


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, C=0.01),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    pred = model.predict(X_test_sc)
    proba = model.predict_proba(X_test_sc)[:, 1]

    print("=" * 50)
    print(name)
    print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))
    print("PR-AUC  :", round(average_precision_score(y_test, proba), 3))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))

Logistic Regression
ROC-AUC : 0.63
PR-AUC  : 0.131
Confusion matrix:
 [[240  53]
 [ 16   5]]
              precision    recall  f1-score   support

           0      0.938     0.819     0.874       293
           1      0.086     0.238     0.127        21

    accuracy                          0.780       314
   macro avg      0.512     0.529     0.500       314
weighted avg      0.881     0.780     0.824       314

Random Forest
ROC-AUC : 0.733
PR-AUC  : 0.17
Confusion matrix:
 [[293   0]
 [ 21   0]]
              precision    recall  f1-score   support

           0      0.933     1.000     0.965       293
           1      0.000     0.000     0.000        21

    accuracy                          0.933       314
   macro avg      0.467     0.500     0.483       314
weighted avg      0.871     0.933     0.901       314



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
from xgboost import XGBClassifier

spw = (y == 0).sum() / (y == 1).sum()   # fail ku adhiga weight

pipes = {
    "Logistic Regression": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=0.01))]),
    "Random Forest": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                       min_samples_leaf=3, random_state=42, n_jobs=-1))]),
    "XGBoost": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.5,
                              scale_pos_weight=spw, eval_metric="logloss",
                              random_state=42))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = {}

for name, pipe in pipes.items():
    proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    oof[name] = proba
    print(f"{name:20s} ROC-AUC: {roc_auc_score(y, proba):.3f} | PR-AUC: {average_precision_score(y, proba):.3f}")

# Best model ku threshold tuning
best = max(oof, key=lambda n: average_precision_score(y, oof[n]))
print("\nBest model:", best)
p, r, t = precision_recall_curve(y, oof[best])
for target in [0.5, 0.6, 0.7]:
    i = np.where(r[:-1] >= target)[0][-1]
    print(f"Recall >= {target}: threshold = {t[i]:.3f}, precision = {p[i]:.3f}")

Logistic Regression  ROC-AUC: 0.660 | PR-AUC: 0.128
Random Forest        ROC-AUC: 0.742 | PR-AUC: 0.154
XGBoost              ROC-AUC: 0.696 | PR-AUC: 0.164

Best model: XGBoost
Recall >= 0.5: threshold = 0.072, precision = 0.133
Recall >= 0.6: threshold = 0.060, precision = 0.138
Recall >= 0.7: threshold = 0.033, precision = 0.106


In [ ]:
for s in ["59", "33", "130", "19", "519"]:
    col = X[s]
    print(f"Sensor {s}: unique = {col.nunique()}, missing = {col.isnull().mean()*100:.1f}%")

Sensor 59: unique = 1500, missing = 0.4%
Sensor 33: unique = 330, missing = 0.1%
Sensor 130: unique = 310, missing = 0.6%
Sensor 19: unique = 361, missing = 0.6%
Sensor 519: unique = 836, missing = 45.6%


In [ ]:
for s in ["33", "130", "19"]:
    print(s, X[s].quantile([0, .01, .05, .5, .95, 1]).round(3).tolist())

m = X["519"].isnull()
print("Sensor 519 missing: fail rate =", round(y[m].mean()*100, 1), "%")
print("Sensor 519 present: fail rate =", round(y[~m].mean()*100, 1), "%")

33 [7.603, 8.117, 8.287, 8.77, 9.714, 23.345]
130 [0.42, 0.544, 0.583, 0.759, 0.853, 0.888]
19 [9.877, 12.127, 12.363, 12.5, 12.611, 12.99]
Sensor 519 missing: fail rate = 4.6 %
Sensor 519 present: fail rate = 8.3 %
